<a href="https://colab.research.google.com/github/stefkong1982/netology.ru/blob/Master/%D0%90%D0%BD%D0%B0%D0%BB%D0%B8%D0%B7_%D1%81%D1%8B%D1%80%D1%8B%D1%85__%D0%A7%D0%B0%D1%81%D1%82%D1%8C_1_ipynb%22.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Цель работы:**  
Разработать модель, способную предсказывать состав следующего заказа пользователя на основе анализа его истории покупок. Это позволит повысить персонализацию сервиса и улучшить пользовательский опыт, а также оптимизировать процессы формирования корзины и планирования закупок.


# Введение и постановка задачи

Современные сервисы доставки продуктов активно используют данные о покупках пользователей для улучшения качества и персонализации сервиса. Анализ истории заказов позволяет прогнозировать, какие категории товаров пользователь может захотеть приобрести в следующий раз.

Целью данного проекта является создание модели, способной по истории заказов прогнозировать состав будущего заказа пользователя. Такая рекомендация поможет клиенту сэкономить время на формировании корзины, избежать забытых позиций и повысить удобство планирования закупок.

Задача формализована как многоклассовая классификация: необходимо предсказать для каждой пары (пользователь, категория), будет ли категория включена в следующий заказ. Для оценки качества модели используется метрика F1-score, которая учитывает баланс между точностью и полнотой предсказаний.

Данные и постановка задачи основаны на открытом соревновании в области электронных продаж.


# Описание набора данных

В проекте используется история заказов 20 000 пользователей, разделённая на тренировочную и тестовую выборки по дате. Тестовая выборка содержит заказы после определённой даты отсечки.

Основной тренировочный файл содержит следующие данные:  
- **user_id** — уникальный идентификатор пользователя  
- **order_completed_at** — дата и время завершения заказа  
- **cart** — категория товара, входящего в заказ (уникальные категории)

Задача — для каждой пары (пользователь, категория), встречающейся в тестовой выборке, предсказать бинарный признак: будет ли категория присутствовать в следующем заказе пользователя.

Идентификаторы пар представлены в формате `"{user_id};{category_id}"`, что учитывается при обработке данных.

Данные подготовлены на основе истории заказов с учётом особенностей временного разделения выборок.


In [1]:
# Подключение Google Drive для доступа к данным и сохранения результатов
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [2]:
# Создание структуры проекта
import os
from pathlib import Path
import pandas as pd
import numpy as np


# Основная папка проекта (как вы указали)
project_path = "/content/drive/MyDrive/Colab Notebooks/sm"

# Чек-лист папок и файлов
structure = [
    "data/raw",          # Исходные данные
    "data/processed",    # Очищенные данные
    "notebooks",         # Jupyter-тетради
    "src/models",        # Код моделей
    "src/utils",         # Утилиты
    "scripts",           # Скрипты обработки
]

# Создаём все директории
for folder in structure:
    os.makedirs(os.path.join(project_path, folder), exist_ok=True)


In [3]:
# Функция для рекурсивного вывода структуры папок проекта
def print_tree(root, prefix=""):
    files = sorted(os.listdir(root))
    for i, name in enumerate(files):
        path = os.path.join(root, name)
        is_last = (i == len(files) - 1)
        branch = "└── " if is_last else "├── "
        print(prefix + branch + name + ("/" if os.path.isdir(path) else ""))
        if os.path.isdir(path):
            new_prefix = prefix + ("    " if is_last else "│   ")
            print_tree(path, new_prefix)

print("\n=== Структура проекта (project_path) ===")
print_tree(project_path)


=== Структура проекта (project_path) ===
├── data/
│   ├── processed/
│   │   └── full_orders.parquet
│   └── raw/
│       ├── sample_submission.csv
│       └── train.csv
├── notebooks/
├── plan/
├── scripts/
└── src/
    ├── models/
    └── utils/


In [4]:
# Блок: Загрузка и первичный анализ train.csv
import pandas as pd
import os

# Путь к папке с данными
project_path = "/content/drive/MyDrive/Colab Notebooks/sm"
raw_data_path = os.path.join(project_path, "data/raw")

# Загрузка данных
train = pd.read_csv(os.path.join(raw_data_path, "train.csv"))
sub = pd.read_csv(os.path.join(raw_data_path, "sample_submission.csv"))

In [5]:
# Информация о train
print("Информация о train.csv:")
print(train.info())
print(f"Уникальных пользователей: {train['user_id'].nunique()}")
print(f"Уникальных категорий: {train['cart'].nunique()}")

Информация о train.csv:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3123064 entries, 0 to 3123063
Data columns (total 3 columns):
 #   Column              Dtype 
---  ------              ----- 
 0   user_id             int64 
 1   order_completed_at  object
 2   cart                int64 
dtypes: int64(2), object(1)
memory usage: 71.5+ MB
None
Уникальных пользователей: 20000
Уникальных категорий: 881


In [6]:
train

,user_id,order_completed_at,cart
0,2,2015-03-22 09:25:46,399
1,2,2015-03-22 09:25:46,14
2,2,2015-03-22 09:25:46,198
3,2,2015-03-22 09:25:46,88
4,2,2015-03-22 09:25:46,157
...,...,...,...
3123059,12702,2020-09-03 23:45:45,441
3123060,12702,2020-09-03 23:45:45,92
3123061,12702,2020-09-03 23:45:45,431
3123062,12702,2020-09-03 23:45:45,24


In [7]:
# Анализ периодов заказов
train['order_completed_at'] = pd.to_datetime(train['order_completed_at'])
print(f"Период данных: с {train['order_completed_at'].min()} по {train['order_completed_at'].max()}")

Период данных: с 2015-03-22 09:25:46 по 2020-09-03 23:45:45


In [8]:
# Анализ sample_submission.csv
print("\nИнформация о sample_submission.csv:")
print(sub.info())


Информация о sample_submission.csv:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 790449 entries, 0 to 790448
Data columns (total 2 columns):
 #   Column  Non-Null Count   Dtype 
---  ------  --------------   ----- 
 0   id      790449 non-null  object
 1   target  790449 non-null  int64 
dtypes: int64(1), object(1)
memory usage: 12.1+ MB
None


In [9]:
# Топ-30 категорий по всему train.csv
from collections import Counter
all_categories = train['cart'].tolist()
cat_counter = Counter(all_categories)
top30 = cat_counter.most_common(30)
top30_cats = [c for c, _ in top30]
print("\nТоп-30 категорий по всем пользователям:", top30_cats)


Топ-30 категорий по всем пользователям: [57, 14, 61, 398, 23, 84, 22, 409, 17, 402, 55, 383, 420, 382, 19, 430, 41, 169, 425, 16, 9, 88, 384, 42, 395, 5, 432, 54, 89, 29]


In [10]:
# Блок: Дополнительная статистика по train.csv

print("\nДополнительная статистика по train.csv:")

orders_per_user = train['user_id'].value_counts()
print("\nРаспределение количества заказов на пользователя:")
print(orders_per_user.describe())



Дополнительная статистика по train.csv:

Распределение количества заказов на пользователя:
count    20000.000000
mean       156.153200
std        200.840781
min          3.000000
25%         48.000000
50%         88.000000
75%        181.000000
max       3508.000000
Name: count, dtype: float64


In [11]:
# Блок: Анализ временных периодов (помесячно)
print("\n=== Анализ временных периодов (помесячно) ===\n")

# Преобразуем дату и создаем месяц-год для группировки
train['order_completed_at'] = pd.to_datetime(train['order_completed_at'])
train['year_month'] = train['order_completed_at'].dt.to_period('M')

# Группируем по месяцам и анализируем
monthly_stats = train.groupby('year_month').agg({
    'order_completed_at': ['count', 'min', 'max'],
    'user_id': 'nunique'
}).round(0)

monthly_stats.columns = ['total_orders', 'first_order', 'last_order', 'unique_users']

# Добавляем информацию о стабильности (проверяем есть ли пропуски дней)
monthly_stats['days_in_month'] = monthly_stats['first_order'].dt.days_in_month

# Правильный способ получить количество уникальных дней с заказами
actual_days = train.groupby('year_month')['order_completed_at'].apply(lambda x: x.dt.date.nunique())
monthly_stats['actual_days'] = actual_days
monthly_stats['missing_days'] = monthly_stats['days_in_month'] - monthly_stats['actual_days']

monthly_stats['stability'] = monthly_stats['missing_days'].apply(
    lambda x: 'стабилен' if x == 0 else f'нестабилен ({int(x)} пропусков)'
)

# Форматируем вывод
for index, row in monthly_stats.iterrows():
    month_str = index.strftime('%Y-%m')
    start_date = row['first_order'].strftime('%Y-%m-%d %H:%M:%S')
    end_date = row['last_order'].strftime('%Y-%m-%d %H:%M:%S')

    print(f"Период {month_str}: {start_date} по {end_date} - {row['stability']} - {int(row['total_orders']):,} заказов")

print(f"\nВсего периодов: {len(monthly_stats)}")
print(f"Общее количество заказов: {monthly_stats['total_orders'].sum():,}")
print(f"Среднее количество заказов в месяц: {monthly_stats['total_orders'].mean():.0f}")


=== Анализ временных периодов (помесячно) ===

Период 2015-03: 2015-03-22 09:25:46 по 2015-03-22 09:25:46 - нестабилен (30 пропусков) - 16 заказов
Период 2015-06: 2015-06-18 16:15:33 по 2015-06-18 16:15:33 - нестабилен (29 пропусков) - 1 заказов
Период 2015-07: 2015-07-04 14:05:22 по 2015-07-22 21:26:56 - нестабилен (28 пропусков) - 17 заказов
Период 2015-08: 2015-08-12 10:33:44 по 2015-08-12 10:33:44 - нестабилен (30 пропусков) - 2 заказов
Период 2015-11: 2015-11-27 19:37:17 по 2015-11-27 19:37:17 - нестабилен (29 пропусков) - 1 заказов
Период 2015-12: 2015-12-01 14:30:59 по 2015-12-14 10:30:14 - нестабилен (29 пропусков) - 15 заказов
Период 2016-04: 2016-04-03 12:03:59 по 2016-04-24 15:59:48 - нестабилен (27 пропусков) - 21 заказов
Период 2016-05: 2016-05-11 13:38:25 по 2016-05-27 19:21:46 - нестабилен (28 пропусков) - 36 заказов
Период 2016-06: 2016-06-04 14:25:03 по 2016-06-07 13:51:38 - нестабилен (28 пропусков) - 10 заказов
Период 2016-07: 2016-07-01 14:51:10 по 2016-07-30 13:11

In [12]:
# Блок: Сравнение топ-30 категорий за весь период и за август 2020
print("\n=== Сравнение топ-30 категорий за весь период и за август 2020 ===\n")

# Топ-30 за весь период (уже есть)
print("Топ-30 категорий за ВЕСЬ период:", top30_cats)

# Топ-30 за август 2020
august_2020_data = train[train['order_completed_at'].dt.to_period('M') == '2020-08']
august_categories = august_2020_data['cart'].tolist()
august_cat_counter = Counter(august_categories)
august_top30 = august_cat_counter.most_common(30)
august_top30_cats = [c for c, _ in august_top30]

print("Топ-30 категорий за АВГУСТ 2020:", august_top30_cats)

# Находим пересечение
common_categories = set(top30_cats) & set(august_top30_cats)
print(f"\nПересечение топ-30: {len(common_categories)} общих категорий")
print("Общие категории:", sorted(common_categories))

# Находим уникальные для всего периода
unique_to_full = set(top30_cats) - set(august_top30_cats)
print(f"\nУникальные для ВСЕГО периода: {len(unique_to_full)} категорий")
print("Уникальные категории:", sorted(unique_to_full))

# Находим уникальные для августа 2020
unique_to_august = set(august_top30_cats) - set(top30_cats)
print(f"\nУникальные для АВГУСТА 2020: {len(unique_to_august)} категорий")
print("Уникальные категории:", sorted(unique_to_august))

# Анализ позиций в рейтинге
print(f"\n=== Анализ позиций в рейтинге ===\n")
rank_comparison = []
for i, (cat, count) in enumerate(top30, 1):
    august_rank = None
    for j, (aug_cat, aug_count) in enumerate(august_top30, 1):
        if cat == aug_cat:
            august_rank = j
            break
    rank_comparison.append((cat, i, august_rank, count))

print("Категория | Ранг(весь период) | Ранг(август 2020) | Количество(весь период)")
print("-" * 70)
for cat, full_rank, aug_rank, count in rank_comparison:
    aug_rank_str = str(aug_rank) if aug_rank else "нет в топ-30"
    print(f"{cat:9} | {full_rank:17} | {aug_rank_str:16} | {count:>24,}")


=== Сравнение топ-30 категорий за весь период и за август 2020 ===

Топ-30 категорий за ВЕСЬ период: [57, 14, 61, 398, 23, 84, 22, 409, 17, 402, 55, 383, 420, 382, 19, 430, 41, 169, 425, 16, 9, 88, 384, 42, 395, 5, 432, 54, 89, 29]
Топ-30 категорий за АВГУСТ 2020: [57, 14, 61, 398, 23, 84, 22, 17, 409, 55, 402, 430, 382, 383, 19, 420, 41, 16, 425, 169, 9, 88, 42, 432, 384, 395, 5, 54, 82, 388]

Пересечение топ-30: 28 общих категорий
Общие категории: [5, 9, 14, 16, 17, 19, 22, 23, 41, 42, 54, 55, 57, 61, 84, 88, 169, 382, 383, 384, 395, 398, 402, 409, 420, 425, 430, 432]

Уникальные для ВСЕГО периода: 2 категорий
Уникальные категории: [29, 89]

Уникальные для АВГУСТА 2020: 2 категорий
Уникальные категории: [82, 388]

=== Анализ позиций в рейтинге ===

Категория | Ранг(весь период) | Ранг(август 2020) | Количество(весь период)
----------------------------------------------------------------------
       57 |                 1 | 1                |                  108,877
       14 |    

In [13]:
# Блок: Анализ продаж категорий 89 и 29 за последний год (с августа 2019 по август 2020)
print("\n=== Анализ продаж категорий 89 и 29 за последний год (с августа 2019 по август 2020) ===\n")

# Фильтруем данные за последний год до августа 2020
start_date = '2019-08-01'
end_date = '2020-08-31'
last_year_data = train[(train['order_completed_at'] >= start_date) &
                       (train['order_completed_at'] <= end_date)].copy()  # Добавляем .copy()

# Группируем по месяцам и категориям 89, 29
last_year_data.loc[:, 'month'] = last_year_data['order_completed_at'].dt.to_period('M')  # Используем .loc
monthly_sales_29_89 = last_year_data[last_year_data['cart'].isin([29, 89])].groupby(['month', 'cart']).size().unstack(fill_value=0)

print("Месяц      | Кат.29 | Кат.89")
print("----------------------------")
for month in monthly_sales_29_89.index:
    cat29 = monthly_sales_29_89.loc[month, 29]
    cat89 = monthly_sales_29_89.loc[month, 89]
    print(f"{month} | {cat29:6} | {cat89:6}")

# Проверяем категории 82 и 388 (уникальные для августа)
print(f"\n=== Анализ категорий 82 и 388 (уникальные для августа) ===\n")
monthly_sales_82_388 = last_year_data[last_year_data['cart'].isin([82, 388])].groupby(['month', 'cart']).size().unstack(fill_value=0)

print("Месяц      | Кат.82 | Кат.388")
print("-----------------------------")
for month in monthly_sales_82_388.index:
    cat82 = monthly_sales_82_388.loc[month, 82] if 82 in monthly_sales_82_388.columns else 0
    cat388 = monthly_sales_82_388.loc[month, 388] if 388 in monthly_sales_82_388.columns else 0
    print(f"{month} | {cat82:6} | {cat388:6}")


=== Анализ продаж категорий 89 и 29 за последний год (с августа 2019 по август 2020) ===

Месяц      | Кат.29 | Кат.89
----------------------------
2019-08 |    265 |    314
2019-09 |    454 |    547
2019-10 |   1083 |   1175
2019-11 |   1457 |   1537
2019-12 |   1319 |   1420
2020-01 |   1228 |   1298
2020-02 |   1192 |   1367
2020-03 |   1646 |   1977
2020-04 |   2290 |   2457
2020-05 |   2981 |   3173
2020-06 |   4304 |   3710
2020-07 |   4199 |   3764
2020-08 |   3755 |   3771

=== Анализ категорий 82 и 388 (уникальные для августа) ===

Месяц      | Кат.82 | Кат.388
-----------------------------
2019-08 |    282 |    232
2019-09 |    484 |    429
2019-10 |   1150 |    901
2019-11 |   1570 |   1165
2019-12 |   1261 |   1013
2020-01 |   1137 |    987
2020-02 |   1211 |   1104
2020-03 |   1661 |   1454
2020-04 |   2165 |   2011
2020-05 |   3014 |   2820
2020-06 |   3717 |   6039
2020-07 |   4047 |   3909
2020-08 |   3899 |   3908


In [28]:
import pandas as pd
import numpy as np

train['order_completed_at'] = pd.to_datetime(train['order_completed_at'])

# Определяем периоды активности
periods = [
    ('12 месяцев', '2019-10-01', '2020-08-31'),
    ('6 месяцев', '2020-03-01', '2020-08-31'),
    ('3 месяца', '2020-06-01', '2020-08-31')
]

# Словарь для хранения результатов
results = {}

# Цикл по каждому периоду
for period_label, start_date, end_date in periods:
    print(f"\n{'-' * 50}\nАнализ периода: {period_label} ({start_date} - {end_date})\n")

    # Фильтруем данные по периоду
    mask = (train['order_completed_at'] >= start_date) & (train['order_completed_at'] <= end_date)
    filtered_train = train.loc[mask]

    # Определяем активных пользователей
    filtered_train.set_index('order_completed_at', inplace=True)
    active_users = (
        filtered_train
        .groupby('user_id')
        .resample('MS')
        .size()
        .reset_index(name='counts')
    )
    active_users_pivot = active_users.pivot(index='user_id', columns='order_completed_at', values='counts')
    active_users_list = active_users_pivot.dropna(axis=0, how='any').index.tolist()

    print(f"Количество активных пользователей: {len(active_users_list)}\n")

    # Создаем словарь для хранения стандартных отклонений
    std_results = {}

    # Рассчитываем стандартное отклонение размера корзины для каждого пользователя
    for user_id in active_users_list:
        user_data = filtered_train[filtered_train['user_id'] == user_id]

        # Размер корзины (количество уникальных категорий в заказе)
        basket_sizes = user_data.groupby('order_completed_at')['cart'].nunique()
        std_basket_size = basket_sizes.std()

        std_results[user_id] = std_basket_size

    # Формируем DataFrame с результатами
    std_df = pd.DataFrame(list(std_results.items()), columns=['user_id', 'Basket_Size_STD'])

    # Определяем медиану стандартного отклонения
    median_std = std_df['Basket_Size_STD'].median()

    # Определяем стабильных пользователей
    stable_users = std_df[std_df['Basket_Size_STD'] <= median_std]

    # Выводим результаты
    print(f"Медианное стандартное отклонение размера корзины: {median_std:.4f}")
    print(f"Количество стабильных пользователей: {len(stable_users)}")
    print(f"Примеры стабильных пользователей и их стандартное отклонение:")
    print(stable_users.head())

    # Сохраняем результаты
    results[period_label] = {
        'active_users': len(active_users_list),
        'median_std': median_std,
        'stable_users': len(stable_users)
    }

# Вывод итоговой таблицы
print("\n{'-' * 50}\nИтоговая таблица распределения пользователей по уровням стабильности:\n")

# Создаем DataFrame для итоговой таблицы
final_table = pd.DataFrame({
    'Период активности': list(results.keys()),
    'Всего активных пользователей': [info['active_users'] for info in results.values()],
    'Медианное стандартное отклонение': [info['median_std'] for info in results.values()],
    'Количество стабильных пользователей': [info['stable_users'] for info in results.values()]
})

print(final_table.to_markdown())


--------------------------------------------------
Анализ периода: 12 месяцев (2019-10-01 - 2020-08-31)

Количество активных пользователей: 1871

Медианное стандартное отклонение размера корзины: 5.8930
Количество стабильных пользователей: 936
Примеры стабильных пользователей и их стандартное отклонение:
    user_id  Basket_Size_STD
5        38         2.528654
9        59         3.132016
15       75         3.872439
19       85         2.187885
21       91         3.576930

--------------------------------------------------
Анализ периода: 6 месяцев (2020-03-01 - 2020-08-31)

Количество активных пользователей: 3880

Медианное стандартное отклонение размера корзины: 5.2071
Количество стабильных пользователей: 1940
Примеры стабильных пользователей и их стандартное отклонение:
    user_id  Basket_Size_STD
1        12         4.195235
2        16         5.024080
6        30         4.509250
7        38         2.887764
15       75         4.092841

-------------------------------------